In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from statsmodels.tsa.api import VAR

# ============================================================
# TVP-VAR (Kalman Filter 기반)
# - 입력: transformed data
# - lag: BIC 선택
# - 결과: time-varying beta, residual cov
# ============================================================

# -----------------------------
# Config
# -----------------------------
BASE_DIR = Path("./")

INPUT_FILE = BASE_DIR / "tvpvar_input_scaled.csv"

OUT_BETA = BASE_DIR / "tvpvar_beta.npy"
OUT_COV = BASE_DIR / "tvpvar_cov.npy"
OUT_LAG = BASE_DIR / "tvpvar_selected_lag.txt"

# Kalman 설정 (논문에서 보통 사용하는 수준)
LAMBDA = 0.99   # state persistence
DELTA = 0.01    # observation noise scale

# -----------------------------
# Load
# -----------------------------
df = pd.read_csv(INPUT_FILE)

# 날짜 제거
if "Date" in df.columns:
    df = df.drop(columns=["Date"])

# 사용할 변수 (ADF 통과 기준)
cols = [
    "dlog_SOLVPN",
    "dlog_COPPER",
    "dlog_GOLD",
    "dlog_SILVER",
    "dlog_DXY",
    "d_UST10Y",
    "d_VIX"
]

df = df[cols].dropna().reset_index(drop=True)

Y = df.values
T, k = Y.shape

print("Data shape:", Y.shape)

# -----------------------------
# Step 1. Lag selection (BIC)
# -----------------------------
var_model = VAR(df)
lag_order = var_model.select_order(maxlags=10)
p = lag_order.selected_orders["bic"]

print("Selected lag (BIC):", p)

with open(OUT_LAG, "w") as f:
    f.write(str(p))

# -----------------------------
# Step 2. Prepare matrices
# -----------------------------
def create_lagged_matrix(Y, p):
    T, k = Y.shape
    X = []

    for t in range(p, T):
        row = []
        for i in range(1, p+1):
            row.extend(Y[t-i])
        X.append(row)

    return np.array(X)

X = create_lagged_matrix(Y, p)
Y_trim = Y[p:]

T_eff = Y_trim.shape[0]
k_beta = k * p

# -----------------------------
# Step 3. Kalman Filter
# -----------------------------
beta_t = np.zeros((T_eff, k, k_beta))
P_t = np.eye(k_beta) * 0.1

Q = np.eye(k_beta) * (1 - LAMBDA)
R = np.eye(k) * DELTA

cov_t = np.zeros((T_eff, k, k))

beta_prev = np.zeros((k, k_beta))

for t in range(T_eff):

    x_t = X[t].reshape(1, -1)
    y_t = Y_trim[t].reshape(-1, 1)

    # ---------------------
    # Prediction step
    # ---------------------
    beta_pred = beta_prev
    P_pred = P_t + Q

    # ---------------------
    # Measurement step
    # ---------------------
    H = np.kron(np.eye(k), x_t)

    y_pred = H @ beta_pred.flatten().reshape(-1,1)
    S = H @ P_pred @ H.T + R

    K = P_pred @ H.T @ np.linalg.inv(S)

    beta_vec = beta_pred.flatten().reshape(-1,1) + K @ (y_t - y_pred)

    beta_new = beta_vec.reshape(k, k_beta)

    P_t = (np.eye(k_beta) - K @ H) @ P_pred

    beta_prev = beta_new

    beta_t[t] = beta_new

    # residual covariance (단순 추정)
    res = y_t - y_pred
    cov_t[t] = res @ res.T

    if t % 200 == 0:
        print(f"t={t}/{T_eff}")

# -----------------------------
# Save
# -----------------------------
np.save(OUT_BETA, beta_t)
np.save(OUT_COV, cov_t)

print("\nSaved:")
print(" -", OUT_BETA)
print(" -", OUT_COV)

Data shape: (1036, 7)
Selected lag (BIC): 0
t=0/1036
t=200/1036
t=400/1036
t=600/1036
t=800/1036
t=1000/1036

Saved:
 - tvpvar_beta.npy
 - tvpvar_cov.npy
